# **Final Notebook: Handcrafted Window Features**

In [1]:
#imports
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

if os.getcwd().endswith("notebooks_final"):
    os.chdir("../")
from src_final.features.local_feature_extractor import WindowFeatureExtractor
from src_final.features.aggregation import aggregate_window_features
from src_final.models.analysis import leakage_free_residual_analysis

In [2]:
processed_path = "data/processed/landmark_dataframes/"
paths = [os.path.join(processed_path, f) for f in os.listdir(processed_path) if f.endswith("30fps_processed.pkl")]

df_dict = {}
cols_to_keep = ['frame', 'hand_label', 'cx_smooth', 'cy_smooth']

# load df with where first column in csv serves as index
df_vid_name_map = pd.read_csv("data/scores/vid_name_map.csv", index_col=0)

with tqdm(total=len(paths), desc="Loading processed data") as pbar:
    for path in paths:
        vid = os.path.basename(path).replace("_30fps_processed.pkl", "")
        vid = vid.replace("hand_tracking_", "")
        participant_id = df_vid_name_map.loc[vid]['Participant Number']
        if int(participant_id) == 8:
            continue
        df_dict[(vid, int(participant_id))] = pd.read_pickle(path)#[cols_to_keep]
        pbar.update(1)

df_dict = dict(sorted(df_dict.items()))

df_scores = pd.read_csv("data/scores/merged_scores.csv")[['Vid_Name', 'QRS_Overal']]
grs_scores = df_scores.set_index('Vid_Name')['QRS_Overal'].to_dict()

Loading processed data:  97%|█████████▋| 83/86 [00:03<00:00, 22.44it/s]


In [3]:
# 1. Initialize and Prepare Data
extractor = WindowFeatureExtractor(hand="Right", window_sec=1.5, step_sec=0.5, log_transform=False, include_bimanual=False)
df_window_features = extractor.extract_features(df_dict)

Extracting Right Features: 100%|██████████| 83/83 [00:48<00:00,  1.72it/s]


In [4]:
feature_cols = [col for col in df_window_features.columns if col not in ['video_id', 'window_start_frame']]

# !!!might want to apply a log transform prior to video level aggregation to some more features!!!
df_window_features[feature_cols].describe()

,total_path,is_idle,path_ratio,curvature,num_reversals,vel_mean,vel_p90,spatial_spread,log_dim_jerk,ang_vel_mean,ang_vel_std,palm_area_cv
count,66805.000000,66805.000000,66805.000000,66805.000000,66805.000000,66805.000000,66805.000000,66805.000000,6.680500e+04,66805.000000,66805.000000,66805.000000
mean,158.748090,0.095996,5.593937,0.552420,2.345199,105.832060,211.767129,26.701205,1.626995e+05,0.666374,0.936019,0.264347
std,146.390429,0.294588,11.842887,0.449945,1.831621,97.593619,215.903027,33.447463,3.147151e+05,0.848246,1.488702,0.248197
min,3.876668,0.000000,1.009416,0.000000,0.000000,2.584445,4.165599,0.279839,1.031118e+03,0.012433,0.015308,0.002284
25%,51.629718,0.000000,1.657437,0.199669,1.000000,34.419812,65.360279,7.666133,5.094642e+04,0.141644,0.168370,0.052784
50%,113.647418,0.000000,2.690903,0.427324,2.000000,75.764945,146.381519,16.005497,9.979890e+04,0.365893,0.448227,0.173780
75%,220.021096,0.000000,5.537311,0.800945,4.000000,146.680731,278.989377,29.556090,1.806853e+05,0.837948,1.047698,0.444270
max,1311.611813,1.000000,683.892277,5.443752,10.000000,874.407875,2293.410623,374.758455,1.872623e+07,12.414626,18.612265,1.751071


In [5]:
# Usage
df_window_features['is_idle'] = (df_window_features['total_path'] < 15).astype(int)
df_agg = aggregate_window_features(df_window_features)

In [6]:
from scipy.stats import pearsonr, spearmanr

candidate_features = [col for col in df_agg.columns if col not in ['video_id']]
scores = list(grs_scores.values())

#shuffle scores
import random
#random.seed(42)
#random.shuffle(scores)

feature_df = df_agg[candidate_features]
# compute correlation with scores both pearson and spearman
from scipy.stats import pearsonr, spearmanr
features = list(feature_df.columns)
pearson_corrs = []
spearman_corrs = []
for i in range(feature_df.shape[1]):
    pearson_corr, _ = pearsonr(feature_df.iloc[:, i], scores)
    spearman_corr, _ = spearmanr(feature_df.iloc[:, i], scores)
    pearson_corrs.append(pearson_corr)
    spearman_corrs.append(spearman_corr)

print("Top 10 features by Pearson correlation:")
top_pearson_indices = np.argsort(np.abs(pearson_corrs))[::-1][:12]
for idx in top_pearson_indices:
    print(f"{features[idx]}:        Pearson r = {pearson_corrs[idx]:.4f}, Spearman rho = {spearman_corrs[idx]:.4f}")

Top 10 features by Pearson correlation:
num_reversals_std:        Pearson r = 0.3982, Spearman rho = 0.4102
path_ratio_p90:        Pearson r = 0.3922, Spearman rho = 0.4526
palm_area_cv_median:        Pearson r = 0.3713, Spearman rho = 0.3664
idle_prop:        Pearson r = 0.3650, Spearman rho = 0.3161
palm_area_cv_p90:        Pearson r = 0.3247, Spearman rho = 0.3890
curvature_std:        Pearson r = 0.3135, Spearman rho = 0.3121
path_ratio_median:        Pearson r = 0.3023, Spearman rho = 0.2795
palm_area_cv_max:        Pearson r = -0.2914, Spearman rho = -0.3030
curvature_p90:        Pearson r = 0.2866, Spearman rho = 0.3276
palm_area_cv_std:        Pearson r = 0.2668, Spearman rho = 0.3230
path_ratio_std:        Pearson r = 0.2383, Spearman rho = 0.1682
num_reversals_p90:        Pearson r = 0.2322, Spearman rho = 0.2095


In [7]:
# load top feature df and combine with aggregated window features
df_top_features = pd.read_csv("data/metrics/top_features_df.csv")
df_combined = pd.merge(df_top_features, df_agg, left_index=True, right_index=True)

In [10]:
top_features = ['num_reversals_Right', 'total_duration_Left', 'nmu_peaks_Right', 'total_path_Right', 'ldlj_smoothness_Left', 'total_angular_path_Right']

candidate_features = [col for col in df_combined.columns if col not in ['Participant Number', 'Case_Number', 'GRS_Total', 'QRS_Overal', 'velocity_corr', 'video_id']+top_features]

#candidate_features.remove('velocity_ratio')
df_res_leakage_free = leakage_free_residual_analysis(df_combined, top_features, candidate_features, base_features=['velocity_corr'], top_n=5, test_other_pcs=False)
df_res_leakage_free

Analyzing Folds:   0%|          | 0/28 [00:00<?, ?it/s]

Analyzing Folds: 100%|██████████| 28/28 [00:04<00:00,  5.90it/s]


,Feature,Type,Mean_Partial_R2,Std_Partial_R2,Mean_Resid_Corr,Selection_Stability
22,spatial_spread_p90,External,0.097469,0.011750,-0.217795,1.000000
14,palm_area_cv_median,External,0.096389,0.009756,0.292105,1.000000
7,idle_prop,External,0.093149,0.011144,0.267528,1.000000
28,vel_p90_p90,External,0.080640,0.010489,-0.145656,0.821429
20,path_ratio_std,External,0.074178,0.008435,0.229747,0.464286
5,curvature_p90,External,0.071544,0.012474,0.189974,0.250000
25,vel_mean_p90,External,0.067753,0.009229,-0.139626,0.142857
15,palm_area_cv_p90,External,0.067217,0.009361,0.226190,0.142857
18,path_ratio_median,External,0.063321,0.007543,0.220985,0.071429
23,spatial_spread_std,External,0.062257,0.008997,-0.142907,0.071429
